## Please run these cells IN ORDER first! This ensures variables established in earlier cells can be used in later cell functions.
I separated this assignment into 3 coding cells to make it easier to read & digest. There are two other Markdown cells besides this one to answer the questions requiring written answers.

In [9]:
# This list of dictionaries was copied exactly as-is from the assignment instructions.

RAW_INSPECTIONS = [

{"hive_id": "H-01", "inspection_date": "2026-08-01", "temp_f": 84,

"weight_lb": "142.5", "mites": 3, "queen_seen": True, "notes": "calm"},

{"hive_id": "H-01", "inspection_date": "2026-08-15", "temp_f": "89",

"weight_lb": "137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},

{"hive_id": "H-01", "inspection_date": "2026-08-15", "temp_f": "89",

"weight_lb": "137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},

{"hive_id": "h-02", "inspection_date": "2026-08-02", "temp_f": "85F",

"weight_lb": 155.2, "mites": 2, "queen_seen": "yes", "notes": None},

{"hive_id": "H-02", "inspection_date": "08/16/2026", "temp_f": 91,

"weight_lb": " 151.4 ", "mites": 5, "queen_seen": True, "notes": ""},

{"hive_id": "H-02", "inspection_date": "2026-08-32", "temp_f": 92,

"weight_lb": 149.8, "mites": 6, "queen_seen": False, "notes": "bad date"},

{"hive_id": "H-03", "inspection_date": "2026-08-03", "temp_f": None,

"weight_lb": 130.0, "mites": 0, "queen_seen": True, "notes": "cool morning"},

{"hive_id": "H-03", "inspection_date": "2026-08-17", "temp_f": 96,

"weight_lb": None, "mites": 9, "queen_seen": "no", "notes": "light hive"},

{"hive_id": "H-04", "inspection_date": "2026-08-04", "temp_f": 79,

"weight_lb": 160.0, "mites": -1, "queen_seen": True, "notes": "counter reset"},

{"hive_id": "H-04", "inspection_date": "2026-08-18", "temp_f": 81,

"weight_lb": 158.5, "mites": 4, "queen_seen": True, "notes": "calm"},

{"hive_id": " H-05 ", "inspection_date": "2026-08-05", "temp_f": "72",

"weight_lb": "bad", "mites": 1, "queen_seen": True, "notes": "scale failed"},

{"hive_id": "H-05", "inspection_date": "2026-08-19", "temp_f": 74,

"weight_lb": 144.2, "mites": 3, "queen_seen": True, "notes": "retest"},

{"hive_id": "H-06", "inspection_date": "2026-08-06", "temp_f": 101,

"weight_lb": 120.0, "mites": 12, "queen_seen": False, "notes": "agitated"},

{"hive_id": "H-06", "inspection_date": "2026-08-20", "temp_f": 97,

"weight_lb": 109.4, "mites": 14, "queen_seen": False, "notes": "low stores"},

]

# This prints the length of the list of records, just so I know how many records there are.
print(f"The length of the original RAW_INSPECTIONS list is {len(RAW_INSPECTIONS)}.")

The length of the original RAW_INSPECTIONS list is 14.


In [11]:
# QUESTION 2: Normalize one record

# This makes a copy of the RAW_INSPECTIONS list of records before the normalization process starts.
raw_insp_copy = RAW_INSPECTIONS.copy() # We use this at the end to make sure we didn't alter the original.

from datetime import date, datetime

def valid_hive_id(value): # Helper function
    """
    Checks for H- followed by exactly two digits.
    
    ARGUMENTS:
    value: can be any type, but will return False if not a string. This is the Hive ID we're checking to see if it's properly formatted.

    RETURNS:
    True if the Hive ID is in proper format, False if not.
    """

    if not isinstance(value, str): # Checks if hive_id is a string. If not, returns False.
        return False

    hive_id = value.strip().upper() # Strips whitespace from hive_id string and makes it uppercase.

    if len(hive_id) != 4: # If hive_id isn't exactly 4 characters, returns False.
        return False

    if hive_id[0:2] != "H-": # If the first two characters aren't "H-", returns False.
        return False

    if not hive_id[2].isdigit() or not hive_id[3].isdigit(): # If the last two characters aren't digits, returns False.
        return False

    return hive_id # Returns normalized hive_id if all criteria is met.


def parse_date(value): # Helper function
    """"
    Returns a datetime.date or None if the date is invalid.

    ARGUMENTS:
    value: can be any type. The date you are trying to convert to a datetime.date().

    RETURNS:
    value converted to a datetime.date or None if it's an invalid date.
    """

    # Gen AI was used to learn about the isinstance function, since I wanted to do different things with different data types and
    # wanted to see what could be used besides type().
    if isinstance(value, date): # Checks if the value is already a date, returns it if so.
        return value

    if not isinstance(value, str): # Checks if the value is a string, returns None if it isn't.
        return None

# This tries using YYYY-MM-DD or MM-DD-YYYY, with dashes (-) and slashes (/) to format the string into a datetime.date.
# If it succeeds, then it returns a datetime.date. If all attempts fail, returns None.
    for fmt in ["%m-%d-%Y", "%m/%d/%Y", "%Y-%m-%d", "%Y/%m/%d"]:
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError: # Used Gen AI to learn about ValueError, since I assumed there must be something similar to IFERROR in Excel.
            continue

    return None


def convert_bool(value): # Helper function
    """
    Converts yes/no queen_seen strings to booleans.
    
    ARGUMENTS:
    value: can be any type, but the function will return None if it's not a string or boolean.

    RETURNS:
    value converted into a boolean, or None if conversion fails.
    """

    if isinstance(value, bool): # Checks if value is already boolean. If so, returns it as-is.
        return value

    if isinstance(value, str): # Checks if the value is a string. If so, strips whitespaces and makes it lowercase.
        value = value.strip().lower()

        if value == "yes": # Changes "yes" str to True.
            return True

        if value == "no": # Changes "no" str to False.
            return False

    return None # Returns None if it doesn't return a boolean value with the above code.


def normalize_record(record: dict) -> tuple[dict | None, str]: # This is the main function.
    """
    This function will take a record and normalize it.

    ARGUMENTS:
    record: Must be DICTIONARY. It's the entry from the RAW_INSPECTIONS list of dictionaries that will be normalized.

    RETURNS:
    A TUPLE with two values. The first will be either a DICTIONARY containing the cleaned record or it will be empty (None) because
    the record was rejected. The second value will be a STRING that explains why the record was rejected if it was, or a blank string.
    """

    norm_record = {} # Establishes blank dictionary for the normalized record output.

    #1. Normalize hive_id by stripping whitespace and converting to uppercase. Accepts only "H-" followed by 2 digits.
    hive_id = record.get("hive_id") # Gen AI was used to learn about the get() function to reference elements within a function argument.

    norm_id = valid_hive_id(hive_id) # Calls the valid_hive_id helper function from earlier.

    if norm_id is False:
        return (None, "invalid hive_id") # If hive_id is false, rejects for invalid hive id.

    norm_record["hive_id"] = norm_id # Sets norm_record to normalized hive id.

    #2 Parse inspection_date by accepting YYYY-MM-DD or MM-DD-YYYY and storing as a datetime.date. Impossible dates are rejected.
    inspection_date = parse_date(record.get("inspection_date")) # Uses the parse_date helper function from earlier.

    if inspection_date is None: # If we failed to convert the date to a datetime.date, rejects for invalid date.
        return (None, "invalid inspection_date")

    norm_record["inspection_date"] = inspection_date # Otherwise, sets norm_record to normalized date.

    #3 Normalize temp_f by removing the final "F" and converting to float. None is fine. Reject values outside -20 thru 130.
    temp = record.get("temp_f")

    if temp is None: # If there is no temp, sets norm_record temp to none.
        norm_record["temp_f"] = None
    else:
        try:
            if isinstance(temp, str): # If temp is a string, strips whitespaces.
                temp = temp.strip()

                if temp.upper().endswith("F"): # If temp ends with an F, removes it.
                    temp = temp[:-1].strip()

            temp = float(temp) # Tries to convert temp to a float.

        except (TypeError, ValueError):
            norm_record["temp_f"] = None # If conversion fails, sets temp to none.

        if temp < -20 or temp > 130:
            return (None, "temp_f out of range") # If temp is under -20 or over 130, rejects for being out of range.

        norm_record["temp_f"] = temp # If checks are passed, sets norm_record to normalized temp.

    #4 Normalize weight_lb by converting to float. If this fails, store None but don't reject. Do reject any negative weight.
    weight = record.get("weight_lb")

    try: # Tries to convert weight to a float.
        weight = float(weight)
    except (TypeError, ValueError):
        weight = None # If conversion doesn't work, sets weight to None.

    if weight is not None and weight < 0: # Rejects if weight is negative.
        return (None, "negative weight")

    norm_record["weight_lb"] = weight # Sets norm_record to normalized weight.

    #5 Normalize mites by converting to type int. Reject any negative values.
    mites = record.get("mites")

    try: # Tries to convert mites to type int.
        mites = int(mites)
    except (TypeError, ValueError):
        return (None, "invalid mites") # If conversion doesn't work, rejects for invalid mites.

    if mites < 0:
        return (None, "negative mites") # If mites are negative, rejects for negative mites.

    norm_record["mites"] = mites # If checks pass, sets norm_record mites to normalized mites.

    #6 Normalize queen_seen by converting "yes" and "no" values to booleans. If already a boolean, keep as-is. Reject any other value besides boolean and yes/no.
    queen_seen = convert_bool(record.get("queen_seen")) # Uses convert_bool helper function from earlier.

    if queen_seen is None: # If convert_bool returned None, rejects for invalid queen_seen.
        return (None, "invalid queen_seen")

    norm_record["queen_seen"] = queen_seen # Otherwise, sets norm_record queen_seen to normalized boolean value.

    #7 Normalize notes by converting any missing notes to an ampty string.
    notes = record.get("notes")

    if notes is None: # Converts notes to empty string if it's none.
        notes = ""

    norm_record["notes"] = notes # Sets norm_record notes to normalized notes value.

    return (norm_record,"") # If it's not rejected earlier, this returns the normalized record and an empty reason string.

print(f"See the normalized first record's dictionary here: {normalize_record(RAW_INSPECTIONS[0])}.")
print(f"See the normalized 6th record's rejection output here, since has an impossible date: {normalize_record(RAW_INSPECTIONS[5])}")


# ============ REQUIRED EVIDENCE ====================================

# This checks to see if the original dictionary has changed since it was copied before the function ran.
# I tested it by changing the original dictionary and it works. I have since removed the code that changed the dictionary,
# of course, since this assignment is supposed to keep it the same.
if raw_insp_copy == RAW_INSPECTIONS:
    print("REQUIRED EVIDENCE 1:\nThe original dictionary has not changed.")
else:
    print("The original dictionary was changed!")

See the normalized first record's dictionary here: ({'hive_id': 'H-01', 'inspection_date': datetime.date(2026, 8, 1), 'temp_f': 84.0, 'weight_lb': 142.5, 'mites': 3, 'queen_seen': True, 'notes': 'calm'}, '').
See the normalized 6th record's rejection output here, since has an impossible date: (None, 'invalid inspection_date')
REQUIRED EVIDENCE 1:
The original dictionary has not changed.


## REQUIRED EVIDENCE 2:
### Why does bad weight remain in the dataset as None, but impossible dates cause rejection?

Bad weight can remain in the dataset as None because the rest of the entry can still be useful and compared with the others.
However, an impossible date is rejected because it makes it impossible to compare with other entries. If you want to place
them in chronological order, you can't if one of the dates isn't real! This calls the validity of the entire record into question.

In [13]:
# QUESTION 3: Clean the collection and keep an audit trail

def clean_records(records: list[dict]) -> tuple[list[dict], list[dict]]:
    """
    This function has normalize_record from the previous cell loop thru a whole list of records.

    ARGUMENTS:
    records: must be a LIST of DICTIONARIES. This is the list of records you are cleaning.

    RETURNS:
    A TUPLE with two lists. The first list is all cleaned records that were not rejected. The second list is all
    records that were rejected during the cleaning process.
    """
    cleaned = [] # Establishes blank list for normalized records (the first output from the normalize_record function)
    rejected = [] # Blank list for rejected records.
    seen = [] # Blank list for records the function has "seen". Used to check for duplicates.

    for source_index, record in enumerate(records): # This uses enumerate to iterate thru the list of records, so we can access the index and value.
        #Gen AI was used to learn to use enumerate properly to access index and value.

        # Normalizes the record, assigns normalized record (index 0) to "clean" and reason string (index 1) to "reason"
        clean = normalize_record(record)[0]
        reason = normalize_record(record)[1]

        # If normalization failed (i.e., clean is type None), adds to audit entry dictionary.
        if clean is None:
            audit_entry = {"source_index": source_index} # Adds source index first.

            # Adds hive_id second, if it is available.
            if "hive_id" in record:
                audit_entry["hive_id"] = record["hive_id"]

            audit_entry["reason"] = reason # Adds rejection reason third.

            rejected.append(audit_entry) # Adds the audit entry dictionary to the rejected list.
            continue # If it's rejected here, skips the remaining code (checking for dupliates) since it's not needed.

        # Creates a key using hive_id and inspection_date to look for duplicates.
        key = (clean["hive_id"], clean["inspection_date"])

        # Checks for duplicates in the "seen" list. If it's in there, adds the record to the rejected list.
        if key in seen:
            rejected.append({"source_index": source_index,
                "hive_id": clean["hive_id"],
                "reason": "duplicate inspection"})
            continue # If it's already in seen, skips the rest of this code so it's not added a second time.

        # If the record is not a duplicate, we add it to the "seen" list and the "cleaned" list.
        seen.append(key)
        cleaned.append(clean)

    return (cleaned, rejected) # Returns a tuple with the list of cleaned records and rejected records.


# This prints the total successes and rejections from the clean_records function.
print(f"For the original RAW_INSPECTIONS list, clean_records returns {len(clean_records(RAW_INSPECTIONS)[0])} successfully cleaned records and {len(clean_records(RAW_INSPECTIONS)[1])} rejections.\n")
print(f"The list of cleaned records is as follows: {clean_records(RAW_INSPECTIONS)[0]}.\n")
print(f"The list of rejections is as follows: {clean_records(RAW_INSPECTIONS)[1]}.\n")


def audit_counts(rejected: list[dict]) -> dict:
    """
    This goes through the rejections and creates a dictionary showing how many records were rejected for each reason.

    ARGUMENTS:
    rejected: must be LIST of DICTIONARIES. This is the list of rejected records from the clean_records function.

    RETURNS:
    a DICTIONARY showing rejection reasons as keys and the # of records rejected for each reason as values.
    """

    # This establishes a "counts" dictionary with every rejection reason. Values are set to 0 at the beginning.
    counts = {"invalid hive_id": 0, "invalid inspection_date": 0, "temp_f out of range": 0, "negative weight": 0,
              "invalid mites": 0, "negative mites": 0, "invalid queen_seen": 0, "duplicate inspection": 0}

    for rejection in rejected: # Iterates through each rejection in rejected list.
        reason = rejection["reason"] # Assigns the rejection reason str to the "reason" variable.

        counts[reason] += 1 # Adds one to the value for the rejection reason key.

    return counts # Returns the updated counts dictionary.

print(f"The output of audit_counts for the original RAW_INSPECTIONS list is: {audit_counts(clean_records(RAW_INSPECTIONS)[1])}.\n")

# Below are the 3 specific tests called for by the assignment.

# TEST 1
# Tests what happens if audit_counts has an empty input. All rejection counts are 0.
print(f"TEST 1: The output of function audit_counts with an empty input is: {audit_counts("")}.")

# TEST 2
# Tests what happens if audit counts has two duplicates (i.e., three of the same record).

# This establishes a list with three of the same record. So one is original, and two are duplicates.
TWO_DUPLICATES = [{"hive_id": "H-01", "inspection_date": "2026-08-01", "temp_f": 84,

"weight_lb": "142.5", "mites": 3, "queen_seen": True, "notes": "calm"},
{"hive_id": "H-01", "inspection_date": "2026-08-01", "temp_f": 84,

"weight_lb": "142.5", "mites": 3, "queen_seen": True, "notes": "calm"},
{"hive_id": "H-01", "inspection_date": "2026-08-01", "temp_f": 84,

"weight_lb": "142.5", "mites": 3, "queen_seen": True, "notes": "calm"}]

# The result is that the "duplicate inspection" reason has a count of 2, and everything else is a count of 0.
print(f"TEST 2: The output of function audit_counts with two duplicate records is: {audit_counts(clean_records(TWO_DUPLICATES)[1])}.")

#TEST 3
# Tests what happens if we use a collection where every record will be rejected.

# Establishes a list of 5 records where each is rejected for a different reason.
ALL_REJECTS = [{"hive_id": "X-01", "inspection_date": "2026-08-01", "temp_f": 84,

"weight_lb": "142.5", "mites": 3, "queen_seen": True, "notes": "calm"},

{"hive_id": "H-01", "inspection_date": "2026-67-89", "temp_f": "89",

"weight_lb": "137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},

{"hive_id": "H-01", "inspection_date": "2026-08-15", "temp_f": "89",

"weight_lb": "-137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},

{"hive_id": "h-02", "inspection_date": "2026-08-02", "temp_f": "850F",

"weight_lb": 155.2, "mites": 2, "queen_seen": "yes", "notes": None},

{"hive_id": "H-02", "inspection_date": "08/16/2026", "temp_f": 91,

"weight_lb": " 151.4 ", "mites": 5, "queen_seen": "nuh uh", "notes": ""},]

# The result is that the rejection counts add up to 5, which is the length of the collection of records.
print(f"TEST 3: The output of function audit_counts with a collection of {len(ALL_REJECTS)} records where each record is rejected is: {audit_counts(clean_records(ALL_REJECTS)[1])}.")

For the original RAW_INSPECTIONS list, clean_records returns 11 successfully cleaned records and 3 rejections.

The list of cleaned records is as follows: [{'hive_id': 'H-01', 'inspection_date': datetime.date(2026, 8, 1), 'temp_f': 84.0, 'weight_lb': 142.5, 'mites': 3, 'queen_seen': True, 'notes': 'calm'}, {'hive_id': 'H-01', 'inspection_date': datetime.date(2026, 8, 15), 'temp_f': 89.0, 'weight_lb': 137.0, 'mites': 8, 'queen_seen': False, 'notes': 'queen not found'}, {'hive_id': 'H-02', 'inspection_date': datetime.date(2026, 8, 2), 'temp_f': 85.0, 'weight_lb': 155.2, 'mites': 2, 'queen_seen': True, 'notes': ''}, {'hive_id': 'H-02', 'inspection_date': datetime.date(2026, 8, 16), 'temp_f': 91.0, 'weight_lb': 151.4, 'mites': 5, 'queen_seen': True, 'notes': ''}, {'hive_id': 'H-03', 'inspection_date': datetime.date(2026, 8, 3), 'temp_f': None, 'weight_lb': 130.0, 'mites': 0, 'queen_seen': True, 'notes': 'cool morning'}, {'hive_id': 'H-03', 'inspection_date': datetime.date(2026, 8, 17), 'te

### Why does duplicate removal happen AFTER normalization?

Duplicate removals happen after normalization because they might not be duplicates beforehand. For example, if the only difference
between two records is that one hive ID is "H-01" and the other is "H-01 " with a space at the end, they wouldn't be considered
duplicates before being normalized. But after both hive IDs are normalized to "H-01", they are revealed as duplicates.